In [ ]:
import hockey_scraper 

# This script scrapes NHL data for the 2025 season playoffs and saves it to a CSV file.
hockey_scraper.scrape_season('2025', 'playoffs', True)


AttributeError: module 'hockey_scraper' has no attribute 'scrape_season'

In [3]:
from pynhl import Scoreboard

scores = Scoreboard()
games_today = scores.games()

for game in games_today:
    print(game['homeTeam']['teamName'], 'vs', game['awayTeam']['teamName'])

ImportError: cannot import name 'Scoreboard' from 'pynhl' (c:\Users\Alex\anaconda3\envs\asdf\lib\site-packages\pynhl\__init__.py)

Now = 7 days

In [9]:
import requests
import pandas as pd

url = "https://api-web.nhle.com/v1/scoreboard/now"
res = requests.get(url)
data = res.json()

games = []

for day in data['gamesByDate']:
    for game in day['games']:
        games.append({
            'date': day['date'],
            'home': game['homeTeam']['abbrev'],
            'away': game['awayTeam']['abbrev'],
            'startTime': game['startTimeUTC'],
            'gameState': game['gameState']
        })

df = pd.DataFrame(games)
print(df)


          date home away             startTime gameState
0   2025-04-25  MTL  WSH  2025-04-25T23:00:00Z       OFF
1   2025-04-25  NJD  CAR  2025-04-26T00:00:00Z       OFF
2   2025-04-25  EDM  LAK  2025-04-26T02:00:00Z       OFF
3   2025-04-26  FLA  TBL  2025-04-26T17:00:00Z       OFF
4   2025-04-26  MIN  VGK  2025-04-26T20:00:00Z       OFF
5   2025-04-26  OTT  TOR  2025-04-26T23:00:00Z       OFF
6   2025-04-26  COL  DAL  2025-04-27T01:30:00Z       OFF
7   2025-04-27  STL  WPG  2025-04-27T17:00:00Z       OFF
8   2025-04-27  NJD  CAR  2025-04-27T19:30:00Z       OFF
9   2025-04-27  MTL  WSH  2025-04-27T22:30:00Z       OFF
10  2025-04-27  EDM  LAK  2025-04-28T01:30:00Z       OFF
11  2025-04-28  FLA  TBL  2025-04-28T23:00:00Z       OFF
12  2025-04-28  DAL  COL  2025-04-29T01:30:00Z      LIVE
13  2025-04-29  TOR  OTT  2025-04-29T23:00:00Z       FUT
14  2025-04-29  CAR  NJD  2025-04-29T23:30:00Z       FUT
15  2025-04-29  VGK  MIN  2025-04-30T01:30:00Z       FUT
16  2025-04-29  LAK  EDM  2025-

In [22]:
import requests
import pandas as pd

# Use today or any playoff date
url = "https://api-web.nhle.com/v1/schedule/now"
headers = {'User-Agent': 'Mozilla/5.0'}

res = requests.get(url, headers=headers)

if res.status_code != 200:
    raise Exception(f"Request failed: {res.status_code}")

data = res.json()

all_playoff_games = []

for day in data.get('gameWeek', []):
    date = day['date']
    for game in day.get('games', []):
        if game.get('gameType') == 3:  # Playoff game
            all_playoff_games.append({
                'date': date,
                'homeTeam': game['homeTeam']['abbrev'],
                'awayTeam': game['awayTeam']['abbrev'],
                'startTimeUTC': game['startTimeUTC'],
                'gameState': game['gameState'],
                'homeScore': game['homeTeam'].get('score'),
                'awayScore': game['awayTeam'].get('score'),
                'seriesStatus': game.get('seriesStatus', {}).get('seriesTitle', '') + " - Game " + str(game.get('seriesStatus', {}).get('gameNumberOfSeries', ''))
            })

# Build DataFrame
df = pd.DataFrame(all_playoff_games)

print(f"✅ Fetched {len(df)} playoff games")
print(df)

# Optional: Save
# df.to_csv("playoff_games_2025.csv", index=False)


✅ Fetched 25 playoff games
          date homeTeam awayTeam          startTimeUTC gameState  homeScore  \
0   2025-04-28      FLA      TBL  2025-04-28T23:00:00Z       OFF        4.0   
1   2025-04-28      DAL      COL  2025-04-29T01:30:00Z      LIVE        5.0   
2   2025-04-29      TOR      OTT  2025-04-29T23:00:00Z       FUT        NaN   
3   2025-04-29      CAR      NJD  2025-04-29T23:30:00Z       FUT        NaN   
4   2025-04-29      VGK      MIN  2025-04-30T01:30:00Z       FUT        NaN   
5   2025-04-29      LAK      EDM  2025-04-30T02:00:00Z       FUT        NaN   
6   2025-04-30      WSH      MTL  2025-04-30T23:00:00Z       FUT        NaN   
7   2025-04-30      TBL      FLA  2025-04-30T23:30:00Z       FUT        NaN   
8   2025-04-30      WPG      STL  2025-05-01T01:30:00Z       FUT        NaN   
9   2025-05-01      OTT      TOR  2025-05-01T16:00:00Z       FUT        NaN   
10  2025-05-01      COL      DAL  2025-05-01T16:00:00Z       FUT        NaN   
11  2025-05-01      MIN  

In [28]:
import requests
import pandas as pd
from datetime import datetime

# Start date and end date
start_date = datetime(2025, 4, 19)
end_date = datetime(2025, 4, 28)

# Request only ONCE
url = f"https://api-web.nhle.com/v1/schedule/{start_date.strftime('%Y-%m-%d')}"
res = requests.get(url)

all_games = []

if res.status_code == 200:
    data = res.json()

    for day in data.get('gameWeek', []):
        for game in day.get('games', []):
            if game.get('gameType') == 3:  # Only playoff games
                # Only add games within the date range we want
                game_date = datetime.strptime(day['date'], "%Y-%m-%d")
                if start_date <= game_date <= end_date:
                    all_games.append({
                        'date': day['date'],
                        'homeTeam': game['homeTeam']['abbrev'],
                        'awayTeam': game['awayTeam']['abbrev'],
                        'startTimeUTC': game['startTimeUTC'],
                        'gameState': game['gameState'],
                        'homeScore': game['homeTeam'].get('score'),
                        'awayScore': game['awayTeam'].get('score'),
                        'seriesStatus': game.get('seriesStatus', {}).get('seriesTitle', '') + " - Game " + str(game.get('seriesStatus', {}).get('gameNumberOfSeries', ''))
                    })
else:
    print(f"❌ Failed to fetch games (status {res.status_code})")

# Done collecting
df = pd.DataFrame(all_games)
print(f"✅ Fetched {len(df)} playoff games from {start_date.date()} to {end_date.date()}")
display(df)

# Optional save
# df.to_csv("playoff_games_filtered.csv", index=False)


✅ Fetched 23 playoff games from 2025-04-19 to 2025-04-28


,date,homeTeam,awayTeam,startTimeUTC,gameState,homeScore,awayScore,seriesStatus
0,2025-04-19,WPG,STL,2025-04-19T22:00:00Z,OFF,5,3,1st Round - Game 1
1,2025-04-19,DAL,COL,2025-04-20T00:30:00Z,OFF,1,5,1st Round - Game 1
2,2025-04-20,CAR,NJD,2025-04-20T19:00:00Z,OFF,4,1,1st Round - Game 1
3,2025-04-20,TOR,OTT,2025-04-20T23:00:00Z,OFF,6,2,1st Round - Game 1
4,2025-04-20,VGK,MIN,2025-04-21T02:00:00Z,OFF,4,2,1st Round - Game 1
5,2025-04-21,WSH,MTL,2025-04-21T23:00:00Z,OFF,3,2,1st Round - Game 1
6,2025-04-21,WPG,STL,2025-04-21T23:30:00Z,OFF,2,1,1st Round - Game 2
7,2025-04-21,DAL,COL,2025-04-22T01:30:00Z,OFF,4,3,1st Round - Game 2
8,2025-04-21,LAK,EDM,2025-04-22T02:00:00Z,OFF,6,5,1st Round - Game 1
9,2025-04-22,CAR,NJD,2025-04-22T22:00:00Z,OFF,3,1,1st Round - Game 2


In [29]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# Start from beginning of playoffs
start_date = datetime(2025, 4, 19)
# End at known playoff end (example June 24)
end_date = datetime(2025, 6, 24)

delta = timedelta(days=7)  # move by 7 days (week by week)

all_games = []

current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    url = f"https://api-web.nhle.com/v1/schedule/{date_str}"
    res = requests.get(url)

    if res.status_code == 200:
        data = res.json()

        for day in data.get('gameWeek', []):
            for game in day.get('games', []):
                if game.get('gameType') == 3:  # Playoffs
                    game_date = datetime.strptime(day['date'], "%Y-%m-%d")
                    if start_date <= game_date <= end_date:
                        all_games.append({
                            'date': day['date'],
                            'homeTeam': game['homeTeam']['abbrev'],
                            'awayTeam': game['awayTeam']['abbrev'],
                            'startTimeUTC': game['startTimeUTC'],
                            'gameState': game['gameState'],
                            'homeScore': game['homeTeam'].get('score'),
                            'awayScore': game['awayTeam'].get('score'),
                            'seriesStatus': game.get('seriesStatus', {}).get('seriesTitle', '') + " - Game " + str(game.get('seriesStatus', {}).get('gameNumberOfSeries', ''))
                        })
    else:
        print(f"⚠️ Failed fetching week starting {date_str}")

    current_date += delta  # move one week ahead

# Done
df = pd.DataFrame(all_games)
print(f"✅ Fetched {len(df)} playoff games from {start_date.date()} to {end_date.date()}")
display(df)

# Optional save
# df.to_csv("all_playoff_games_2025.csv", index=False)


✅ Fetched 56 playoff games from 2025-04-19 to 2025-06-24


,date,homeTeam,awayTeam,startTimeUTC,gameState,homeScore,awayScore,seriesStatus
0,2025-04-19,WPG,STL,2025-04-19T22:00:00Z,OFF,5.0,3.0,1st Round - Game 1
1,2025-04-19,DAL,COL,2025-04-20T00:30:00Z,OFF,1.0,5.0,1st Round - Game 1
2,2025-04-20,CAR,NJD,2025-04-20T19:00:00Z,OFF,4.0,1.0,1st Round - Game 1
3,2025-04-20,TOR,OTT,2025-04-20T23:00:00Z,OFF,6.0,2.0,1st Round - Game 1
4,2025-04-20,VGK,MIN,2025-04-21T02:00:00Z,OFF,4.0,2.0,1st Round - Game 1
5,2025-04-21,WSH,MTL,2025-04-21T23:00:00Z,OFF,3.0,2.0,1st Round - Game 1
6,2025-04-21,WPG,STL,2025-04-21T23:30:00Z,OFF,2.0,1.0,1st Round - Game 2
7,2025-04-21,DAL,COL,2025-04-22T01:30:00Z,OFF,4.0,3.0,1st Round - Game 2
8,2025-04-21,LAK,EDM,2025-04-22T02:00:00Z,OFF,6.0,5.0,1st Round - Game 1
9,2025-04-22,CAR,NJD,2025-04-22T22:00:00Z,OFF,3.0,1.0,1st Round - Game 2
